# Pipeline Evaluation — chequeo manual post-clasificación

Objetivo: en 5 minutos verificar que las 3 tablas canónicas tienen sentido.  
Corre cada sección de arriba a abajo. Si algo se ve raro, es el lugar para anotarlo.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 60)

classified = pd.read_csv('../data/processed/reviews_classified.csv')
agg        = pd.read_csv('../data/processed/aggregated.csv')
ins        = pd.read_csv('../data/processed/insights.csv')

has_text   = classified[classified['has_text'] == True].copy()
cl         = classified[classified['sentiment'].notna()].copy()

print(f"reviews_classified : {len(classified):>4} rows  |  classified: {len(cl)}  |  rating-only: {classified['sentiment'].isna().sum()}")
print(f"aggregated         : {len(agg):>4} rows")
print(f"insights           : {len(ins):>4} rows")

reviews_classified :  250 rows  |  classified: 155  |  rating-only: 95
aggregated         :    4 rows
insights           :   27 rows


---
## 1 · Sanidad de datos — conteos y nulos

In [2]:
# Rows per business
classified.groupby('business_name').agg(
    total=('review_id','count'),
    with_text=('has_text','sum'),
    classified=('sentiment', lambda s: s.notna().sum()),
    rating_only=('sentiment', lambda s: s.isna().sum()),
    errors=('sentiment', lambda s: (s == 'error').sum()),
).assign(pct_classified=lambda d: (d['classified'] / d['with_text'] * 100).round(1))

,total,with_text,classified,rating_only,errors,pct_classified
business_name,,,,,,
El Mesón de Gauss,50,30,29,21,0,96.7
IDA Restaurant Bar,100,53,53,47,0,100.0
Mitica,50,40,40,10,0,100.0
Vittorio Ristorante,50,33,33,17,0,100.0


In [3]:
# Duplicate review_ids?
dupes = classified['review_id'].duplicated().sum()
print(f"Duplicate review_ids: {dupes}")

# Nulls in classification fields (should only be on rating-only rows)
cl_fields = ['sentiment','main_topic','subtopic','urgency','is_actionable','classification_confidence']
print("\nNulls per field (classified rows only):")
print(cl[cl_fields].isna().sum())

Duplicate review_ids: 0

Nulls per field (classified rows only):
sentiment                     0
main_topic                    0
subtopic                     21
urgency                       0
is_actionable                 0
classification_confidence     0
dtype: int64


---
## 2 · Distribuciones — sentiment / topic / urgency / confianza

In [4]:
# Sentiment breakdown per business
(
    cl.groupby(['business_name','sentiment'])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda d: d.sum(axis=1))
    .assign(
        pct_pos=lambda d: (d['positive']/d['total']*100).round(1),
        pct_neg=lambda d: (d['negative']/d['total']*100).round(1),
    )
)

sentiment,negative,neutral,positive,total,pct_pos,pct_neg
business_name,,,,,,
El Mesón de Gauss,18,1,10,29,34.5,62.1
IDA Restaurant Bar,13,0,40,53,75.5,24.5
Mitica,6,0,34,40,85.0,15.0
Vittorio Ristorante,5,0,28,33,84.8,15.2


In [5]:
# Topic distribution — global
cl['main_topic'].value_counts()

main_topic
Food Quality              46
Overall Experience        39
Staff Attitude            38
Menu & Options            10
Service Speed              9
Ambiance                   6
Price / Value              4
Hygiene & Cleanliness      2
Booking & Reservations     1
Name: count, dtype: int64

In [6]:
# Urgency distribution
cl['urgency'].value_counts()

urgency
low       110
medium     37
high        8
Name: count, dtype: int64

In [7]:
# Confidence distribution
print(cl['classification_confidence'].describe().round(3))
print(f"\nLow confidence (<0.70): {(cl['classification_confidence'] < 0.70).sum()} reviews")

count    155.000
mean       0.952
std        0.050
min        0.800
25%        0.900
50%        0.950
75%        1.000
max        1.000
Name: classification_confidence, dtype: float64

Low confidence (<0.70): 0 reviews


---
## 3 · Spot-check — muestras aleatorias para revisión manual

Leé cada texto y verificá que el label tenga sentido. Si algo está mal anotalo más abajo.

In [8]:
SAMPLE_N = 20  # cambiá este número si querés ver más
RANDOM_SEED = 42

sample = cl.sample(SAMPLE_N, random_state=RANDOM_SEED)[
    ['business_name','clean_text','sentiment','main_topic','subtopic','urgency','is_actionable','classification_confidence']
].reset_index(drop=True)

sample

,business_name,clean_text,sentiment,main_topic,subtopic,urgency,is_actionable,classification_confidence
0,El Mesón de Gauss,"Mala atención, la comida no nos gustó tampoco, los wraps secos el gohan que no es gohan l ensalada César desabrida",negative,Food Quality,multiple food quality issues,medium,True,0.90
1,Vittorio Ristorante,"excelente atención, exquisitos los tagliatelle con mariscos! el crepe de dulce de leche y el tiramisú muy buenos!! l...",positive,Food Quality,Specific dishes,low,False,0.90
2,IDA Restaurant Bar,"Primera visita y muy grata! La comida riquísima, hay una entrada súper rica. Hasta el pan es casero. La atención un ...",positive,Overall Experience,lighting improvement suggestion,low,True,0.90
3,IDA Restaurant Bar,Excelente. Muy muy recomendable. Una experiencia muy agradable y una comida exquisita.,positive,Overall Experience,pleasant experience and exquisite food,low,False,1.00
4,Mitica,": ""Quiero compartir mi experiencia en Mítica. Hicimos una reserva para 8 personas para el menú ejecutivo el día ante...",positive,Booking & Reservations,Executive menu availability,medium,True,0.90
5,El Mesón de Gauss,"Vinimos con unos amigos, y NO HAY PIZZAS! lo peor es que no te advierte cuando te indican que abras la carta. Y tamp...",negative,Menu & Options,unavailable items and lack of notification,medium,True,0.90
6,Mitica,"Hermoso lugar, primera vez que voy y quedé encantada. La atención de los chicos muy buena y Milagros, la recepcionis...",positive,Staff Attitude,friendly and attentive staff,low,False,0.90
7,Vittorio Ristorante,Fui desde el primer día....y lo sigo eligiendo,positive,Overall Experience,NaN,low,False,1.00
8,Vittorio Ristorante,"Excelente todo !!! Muy rica la comida. Andrés, uno de los mozos, un crack muy amable y gentil. Los felicito.",positive,Staff Attitude,specific staff member praise,low,False,0.85
9,El Mesón de Gauss,"Solo café , pésimo,quemado ,lástima porque trabajan Ardu que es un café excelente",negative,Food Quality,burnt coffee,medium,True,0.95


In [9]:
# LOW CONFIDENCE — revisar estos primero (más propensos a error)
cl[cl['classification_confidence'] < 0.75][
    ['business_name','clean_text','sentiment','main_topic','urgency','classification_confidence']
].sort_values('classification_confidence').head(15)

,business_name,clean_text,sentiment,main_topic,urgency,classification_confidence


In [10]:
# HIGH URGENCY — verificar que realmente sean urgentes
cl[cl['urgency'] == 'high'][
    ['business_name','clean_text','sentiment','main_topic','subtopic','urgency','is_actionable']
]

,business_name,clean_text,sentiment,main_topic,subtopic,urgency,is_actionable
38,IDA Restaurant Bar,"Fuimos anoche y realmente muy decepcionado Desde la mesa, a la moza con la mini copa de bienvenida era tan Poco que ...",negative,Food Quality,"Inedible steak, slow replacement",high,True
51,IDA Restaurant Bar,La comida más o menos y lo peor el servicio dejó muchísimo que desear. Tuvieron un accidente con un plato de bondiol...,negative,Staff Attitude,Lack of apology after incident,high,True
55,IDA Restaurant Bar,"Fuimos un viernes por la noche con mi mamá, planeando una linda ""salida de chicas"", pero lamentablemente terminamos ...",negative,Hygiene & Cleanliness,rotten food causing illness and unsafe practices,high,True
95,IDA Restaurant Bar,"Acotaron mucho la carta, y encima lo que pedíamos no tenían. Mucha demora en traernos las cosas. Casi nada de varied...",negative,Menu & Options,limited selection and unavailability of items,high,True
107,El Mesón de Gauss,"Un horror,fui con mi hija y mi nieta a merendar,dos cafe con leche,tostado tostadas fit,y criollitos,en la mitad de ...",negative,Hygiene & Cleanliness,dead fly in drink / unclean cups,high,True
124,El Mesón de Gauss,Estacionamiento poco cómodo. Buen servicio de las mozas. Muy buenos los sandwichs. Fría la cerveza. Las papas snacks...,negative,Food Quality,rancid snacks,high,True
163,Mitica,Se desentienden del Wp y las redes. Tenés que ir en persona para enterarte . Tienen el menú ejecutivo en una histori...,negative,Service Speed,2-hour wait for lunch,high,True
214,Vittorio Ristorante,"Melhorem o atendimento, fomos atendidos por um rapaz de óculos que sou super grosseiro e impaciente…",negative,Staff Attitude,rude and impatient staff,high,True


In [11]:
# NEGATIVE + ACTIONABLE — los más importantes para el reporte
cl[(cl['sentiment'] == 'negative') & (cl['is_actionable'] == True)][
    ['business_name','clean_text','main_topic','subtopic','urgency','classification_confidence']
].sort_values('urgency', ascending=False)

,business_name,clean_text,main_topic,subtopic,urgency,classification_confidence
29,IDA Restaurant Bar,"No fuimos bien atendidos, nos tomaron la mitad del pedido, al no haber opciones de platos principales vegetarianos, ...",Menu & Options,lack of vegetarian/vegan main courses and reduced menu,medium,0.90
149,El Mesón de Gauss,"Mala atención, la comida no nos gustó tampoco, los wraps secos el gohan que no es gohan l ensalada César desabrida",Food Quality,multiple food quality issues,medium,0.90
129,El Mesón de Gauss,"Anoche después de ir al teatro fuimos a comer al Mesón, pedimos lomito se demoraron una hora en traerlo, no nos ofre...",Service Speed,long wait time,medium,1.00
134,El Mesón de Gauss,No me gustó nada la comida. El lugar es lindo.,Food Quality,taste of food,medium,1.00
135,El Mesón de Gauss,"40 minutos esperando atención, el lugar no estaba lleno, estuvimos afuera una desilusión... Mala atención",Service Speed,Slow service / Waiting time,medium,1.00
136,El Mesón de Gauss,Fuimos hoy porque buscábamos donde desayunar y que tuviesen juegos para niños . encontramos comentarios de que acá t...,Food Quality,thick bread and warm juice,medium,0.85
138,El Mesón de Gauss,Me sirvieron una tabla mesón y vino: -Fria el 90% de los ingredientes salvo las rabas. -Precisamente las rabas eran ...,Food Quality,Food temperature and texture,medium,1.00
147,El Mesón de Gauss,"El lugar muy lindo, la comida buena, la atención hace que no tengas ganas de volver....",Staff Attitude,unwelcoming service,medium,0.90
148,El Mesón de Gauss,Desayune muy bien. El servicio de mozas muy flojo,Staff Attitude,waitress performance,medium,0.90
156,Mitica,Si bien el ambiente es muy lindo la taza del desayuno es muy chica y las medialunas también. No conduce la calidad d...,Food Quality,portion size,medium,0.95


---
## 4 · Aggregated — chequeo de métricas de negocio

In [12]:
agg

,business_name,total_reviews,reviews_with_text,avg_rating,pct_positive,pct_neutral,pct_negative,top_topic_1,top_topic_2,top_topic_3,high_urgency_count,actionable_count
0,El Mesón de Gauss,50,30,3.6,34.5,3.4,62.1,Food Quality,Service Speed,Staff Attitude,2,19
1,IDA Restaurant Bar,100,53,4.4,75.5,0.0,24.5,Overall Experience,Staff Attitude,Food Quality,4,18
2,Mitica,50,40,4.6,85.0,0.0,15.0,Food Quality,Staff Attitude,Overall Experience,1,11
3,Vittorio Ristorante,50,33,4.6,84.8,0.0,15.2,Overall Experience,Food Quality,Staff Attitude,1,8


In [13]:
# Verificar que pct_positive + pct_neutral + pct_negative ≈ 100 en cada fila
agg['pct_check'] = agg['pct_positive'] + agg['pct_neutral'] + agg['pct_negative']
print("pct sum per business (expected ~100):")
print(agg[['business_name','pct_check']])

pct sum per business (expected ~100):
         business_name  pct_check
0    El Mesón de Gauss      100.0
1   IDA Restaurant Bar      100.0
2               Mitica      100.0
3  Vittorio Ristorante      100.0


---
## 5 · Insights — ranking de problemas

In [14]:
# Top issues por negocio — estas son las recomendaciones del reporte
ins.sort_values('priority_score', ascending=False)

,business_name,main_topic,mention_count,pct_negative,avg_urgency_score,actionable_count,priority_score
0,El Mesón de Gauss,Food Quality,11,63.6,1.73,7,12.10
1,El Mesón de Gauss,Service Speed,4,100.0,2.00,4,8.00
2,IDA Restaurant Bar,Menu & Options,5,80.0,2.00,4,8.00
3,Vittorio Ristorante,Service Speed,3,100.0,2.00,3,6.00
4,IDA Restaurant Bar,Staff Attitude,15,26.7,1.40,8,5.61
5,Mitica,Service Speed,2,100.0,2.50,2,5.00
6,El Mesón de Gauss,Menu & Options,2,100.0,2.00,2,4.00
7,El Mesón de Gauss,Price / Value,3,66.7,2.00,3,4.00
8,IDA Restaurant Bar,Food Quality,12,25.0,1.33,3,3.99
9,Mitica,Food Quality,13,23.1,1.23,4,3.69


In [15]:
# Por business — qué problema lidera en cada uno
ins.groupby('business_name').first()[['main_topic','priority_score','pct_negative','avg_urgency_score']]

,main_topic,priority_score,pct_negative,avg_urgency_score
business_name,,,,
El Mesón de Gauss,Food Quality,12.1,63.6,1.73
IDA Restaurant Bar,Menu & Options,8.0,80.0,2.00
Mitica,Service Speed,5.0,100.0,2.50
Vittorio Ristorante,Service Speed,6.0,100.0,2.00


---
## 6 · Hallazgos manuales

Usá esta celda para anotar lo que encontraste en el spot-check:

**Clasificaciones correctas:** 

**Clasificaciones dudosas o incorrectas:** 

**Patrones que el modelo no captó bien:** 

**Acciones a tomar (prompt, taxonomía, etc.):** 